In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

print(y_pred[:10])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("ML MAE:", mae)
print("ML RMSE:", rmse)

In [ ]:
results = pd.DataFrame({
    "Model": ["Naive", "Seasonal Naive", "HistGradientBoosting"],
    "MAE": [1959.73, 920.93, mae],
    "RMSE": [2538.04, 1220.83, rmse]
})

print(results)

In [ ]:
comparison = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred
})

print(comparison.head(20))

In [ ]:
import matplotlib.pyplot as plt

comparison.iloc[:24 * 7].plot(
    figsize=(15, 5)
)

plt.title("Actual vs Predicted")
plt.xlabel("Datetime")
plt.ylabel("AEP_MW")
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=3,
    random_state=42,
    scoring="neg_mean_absolute_error"
)

importance = pd.Series(
    result.importances_mean,
    index=X_test.columns
).sort_values(ascending=False)

print(importance)

In [ ]:
errors = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred
})

errors["error"] = errors["actual"] - errors["predicted"]
errors["abs_error"] = errors["error"].abs()
errors["hour"] = errors.index.hour

mae_by_hour = errors.groupby("hour")["abs_error"].mean()

print(mae_by_hour)

In [ ]:
mae_by_day = errors.groupby(
    errors.index.dayofweek
)["abs_error"].mean()

print(mae_by_day)

In [ ]:
# циклические признаки времени

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

In [ ]:
features = [
    "hour",
    "day_of_week",
    "month",
    "day_of_year",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "lag_1",
    "lag_24",
    "lag_168",
    "rolling_mean_24",
    "rolling_mean_168"
]

In [ ]:
X = df[features]
y = df["AEP_MW"]

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [ ]:
print(mae_by_hour.sort_values(ascending=False))

In [ ]:
print(mae_by_day.sort_values(ascending=False))

In [ ]:
print(importance.sort_values(ascending=False))

In [ ]:
X = df[features].copy()

X["hour_sin"] = np.sin(2 * np.pi * X["hour"] / 24)
X["hour_cos"] = np.cos(2 * np.pi * X["hour"] / 24)

split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

model = HistGradientBoostingRegressor(
    max_iter=300,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print("ML MAE:", mae)
print("ML RMSE:", rmse)

In [ ]:
X = df[features].copy()

X["hour_sin"] = np.sin(2 * np.pi * X["hour"] / 24)
X["hour_cos"] = np.cos(2 * np.pi * X["hour"] / 24)

X["dow_sin"] = np.sin(2 * np.pi * X["day_of_week"] / 7)
X["dow_cos"] = np.cos(2 * np.pi * X["day_of_week"] / 7)

split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

model = HistGradientBoostingRegressor(
    max_iter=300,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print("ML MAE:", mae)
print("ML RMSE:", rmse)

In [ ]:
X = df[features].copy()

# Циклическое время
X["hour_sin"] = np.sin(2 * np.pi * X["hour"] / 24)
X["hour_cos"] = np.cos(2 * np.pi * X["hour"] / 24)

# Дополнительные лаги
X["lag_2"] = y.shift(2)
X["lag_3"] = y.shift(3)
X["lag_6"] = y.shift(6)
X["lag_12"] = y.shift(12)

# Синхронно удаляем строки с NaN
data = X.copy()
data["target"] = y

data = data.dropna()

X = data.drop(columns="target")
y_model = data["target"]

print(X.shape)
print(y_model.shape)

In [ ]:
split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y_model.iloc[:split]
y_test = y_model.iloc[split:]

model = HistGradientBoostingRegressor(
    max_iter=300,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print("ML MAE:", mae)
print("ML RMSE:", rmse)

In [ ]:
X = df[features].copy()

X["hour_sin"] = np.sin(2 * np.pi * X["hour"] / 24)
X["hour_cos"] = np.cos(2 * np.pi * X["hour"] / 24)

X["lag_2"] = y.shift(2)
X["lag_3"] = y.shift(3)
X["lag_6"] = y.shift(6)
X["lag_12"] = y.shift(12)

data = X.copy()
data["target"] = y
data = data.dropna()

X = data.drop(columns="target")
y_model = data["target"]

In [ ]:
errors = pd.DataFrame({
    "actual": y_test,
    "pred": y_pred
})

errors["abs_error"] = abs(errors["actual"] - errors["pred"])
errors["month"] = errors.index.month

print(
    errors.groupby("month")["abs_error"]
    .mean()
    .sort_values(ascending=False)
)

In [ ]:
errors["year"] = errors.index.year

print(
    errors.groupby("year")["abs_error"]
    .mean()
    .sort_values(ascending=False)
)

In [ ]:
print(
    errors.sort_values("abs_error", ascending=False)
    [["actual", "pred", "abs_error"]]
    .head(20)
)

In [ ]:
for threshold in [500, 1000, 1500, 2000, 3000]:
    filtered = errors[errors["abs_error"] <= threshold]
    print(
        f"<= {threshold}: "
        f"MAE = {filtered['abs_error'].mean():.2f}, "
        f"points = {len(filtered)}"
    )

In [ ]:
baseline_pred = X_test["lag_1"]

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = mean_squared_error(y_test, baseline_pred) ** 0.5

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)

print("ML MAE:", mean_absolute_error(y_test, y_pred))
print("ML RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))

plt.plot(y_test.index[:500], y_test.iloc[:500], label="Actual")
plt.plot(y_test.index[:500], y_pred[:500], label="Predicted")

plt.title("Actual vs Predicted Energy Consumption")
plt.xlabel("Datetime")
plt.ylabel("Energy Consumption")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()